#  Extreme Gradient Boosting
This algorithm is one of the most widely used and high-performance machine learning algorithms for predictive modeling across a multiple domains. 
It is an optimized and efficient implementation of the Gradient Boosting framework.


XGBoost is based on boosting, we build multiple weaker models and combine them to get better performance.


In this, each tree learns to predict a correction term (residual), which adjusts the current prediction.
So each tree contributes only a small adjustment to the overall prediction.



### The training workflow 

The models are built sequentially, where each new model focuses on correcting the errors made by the previous ones. Over time, the combined model becomes more accurate by gradually improving on its mistakes.

We begin with a simple prediction (a constant value or the avg of the targets values in the training data), measure where it performs poorly, and then add another model that focuses on those weak areas. This process is repeated many times to create a strong final predictor.

$$
\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + f_t(x_i)
$$
Here, the prediction at step $t$ is the previous prediction plus the contribution of the new tree $f_t$. Each tree is therefore not predicting from scratch; it is correcting what the current model still gets wrong.

The final prediction is a sum of the trees:
$
\hat{y}_i = \sum_{t=1}^{T} f_t(x_i), \quad f_t \in \mathcal{F}$

Each $f_t$ is a regression tree. The leaves of a tree store real-valued scores, not class labels. Those scores are added together across trees to produce the final output.


## The Cost Function and optimisation
 It defines a formal objective made of two parts:

$obj(\theta) = L(\theta) + \Omega(\theta)$

Here, $L$ is the training loss, and $\Omega$ is the regularization term. The loss measures how well predictions match the labels, while the regularization term controls complexity and helps prevent overfitting.

For the $t$-th boosting step, the objective is written as:

$
obj(t) = \sum_{i=1}^{n} l(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t) + constant$

This shows the exact logic of boosting in XGBoost: the new tree ($f_t$) is chosen to improve the current prediction while also accounting for model complexity.



For general loss functions, XGBoost uses a second-order Taylor expansion of the loss around the current prediction. That gives a local approximation involving the first derivative $g_i$ and the second derivative ($h_i$):

$
g_i = \partial_{\hat{y}_i^{(t-1)}} l(y_i, \hat{y}_i^{(t-1)}), \qquad
h_i = \partial_{\hat{y}_i^{(t-1)}}^2 l(y_i, \hat{y}_i^{(t-1)})
$

Using these, the new-tree objective becomes:

$
\sum_{i=1}^{n} \left[g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i)\right] + \Omega(f_t)
$

This is one of the reasons XGBoost is so flexible: the same solver can be used for different tasks, including regression, logistic regression, and ranking, only gradient and Hessian information from the chosen loss is required for model. 



## Regularization in XGBoost

A major reason XGBoost works so well is that it regularizes the tree itself, not only the final prediction error. The complexity term is defined as:

$
\Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2
$

where $T$ is the number of leaves and $w_j$ are the leaf scores. The $\gamma T$ term penalizes too many leaves, and the $\lambda$ term penalizes large leaf values. This is one of the key differences between XGBoost and older tree boosting implementations that relied more heavily on heuristics for complexity control. :contentReference

This regularization is not just a small technical detail. It directly shapes the trees the algorithm prefers. A tree that improves training loss a little but adds too much complexity may be rejected, because the penalty outweighs the gain. That is one of the main reasons XGBoost tends to generalize well.
### How the tree structure is chosen

Once XGBoost has the objective for the new tree, it still has to decide which split to make. It does this greedily(similar to the DT -> [Decision-Tree.ipynb](DecisionTree) ), one split at a time, because trying every possible tree structure is intractable. For each possible split, it computes a gain score:

$
Gain =\frac{1}{2} \left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda} \right] - \gamma$


The gain measures how much the objective improves if a leaf is split into left and right children. If the gain is not large enough to overcome the complexity penalty, the split is not kept. In other words, the method grows the tree only when the split is actually worth the added complexity. 


In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing

cali = fetch_california_housing()
X =cali.data
y =cali.target
X_tr, X_test, y_tr, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
XGB=XGBRegressor()
XGB.fit(X_tr,y_tr)
pred=XGB.predict(X_test)
print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R^2 Score:", r2_score(y_test, pred))



Mean Squared Error: 0.2225899267544737
R^2 Score: 0.8301370561019205


For classification problems, XGBoost still uses regression trees.

## Binary Classification

In binary classification, the model builds a score (also called a logit):

$$
\hat{y} = \sum_{t=1}^{T} f_t(x)
$$

This score represents how strongly the model leans toward one class.

To convert this score into a probability, XGBoost applies the sigmoid function:

$$
P(y=1 \mid x) = \frac{1}{1 + e^{-\hat{y}}}
$$

The final prediction is then obtained by applying a threshold .



The model begins with a constant prediction, typically the log-odds of the positive class computed from the training data:

$$
\hat{y}^{(0)} = \log \left(\frac{p}{1-p}\right)
$$

where $p$ is the proportion of positive samples.

Starting from this initial value, the model adds trees one by one. Each tree learns to correct the current prediction using the gradients of the log loss (cross-entropy).

So instead of predicting classes directly, each tree makes small adjustments to the score in a way that improves the predicted probability.



## Multiclass Classification

For multiclass problems, the model maintains a separate score for each class:

$$
\hat{y}_k = \sum_{t=1}^{T} f_{t,k}(x)
$$

Each class accumulates its own score across trees.

These scores are converted into probabilities using the softmax function:

$$
P(y = k \mid x) = \frac{e^{\hat{y}_k}}{\sum_{j} e^{\hat{y}_j}}
$$

The class with the highest probability is selected as the final prediction.

The model starts with initial scores based on class proportions, so each class begins with a constant value that reflects how frequently it appears in the training data.



## Loss Function in Classification

For classification, XGBoost minimizes a regularized objective:

$$
obj(\theta) = L(\theta) + \Omega(\theta)
$$

- $L(\theta)$ → classification loss  
  - binary: logistic loss  
  - multiclass: softmax loss  

- $\Omega(\theta)$ → regularization term controlling model complexity  

The model updates predictions by reducing the classification loss while keeping the tree structure controlled through regularization.

In [32]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_wine
from sklearn.preprocessing import LabelEncoder
data=load_wine()
X=data.data
y=data.target

Xtr, X_test, ytr, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
XGB_clf=XGBClassifier()
XGB_clf.fit(Xtr,ytr)
pred_clf=XGB_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred_clf))
print("Classification Report:\n", classification_report(y_test, pred_clf))

Accuracy: 0.9444444444444444
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.93      0.96        14
           1       0.88      1.00      0.93        14
           2       1.00      0.88      0.93         8

    accuracy                           0.94        36
   macro avg       0.96      0.93      0.94        36
weighted avg       0.95      0.94      0.94        36



## Other Features: 
THe model also handle missing values internally. During training, it learns the optimal direction (left or right in a split) for missing data points based on which choice minimizes the loss. This allows the model to work effectively even when the dataset contains incomplete entries, without requiring explicit imputation.

XGBoost also supports parallel processing during tree construction for faster training. While boosting itself is sequential (since each tree depends on the previous one), the algorithm parallelizes operations such as split finding and feature evaluation within each tree.